In [ ]:
import tifffile
import numpy as np
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
from skimage import io, color, filters, measure, morphology
from scipy.stats import shapiro, ttest_ind, mannwhitneyu


plt.style.use("dark_background")


path = r""

target = 2
target_name = ""

mask = 1
mask_name = ""
invert_mask = False  # True = analyze outside the mask

second_mask = None
second_mask_name = "" #Channel2
invert_second_mask = False  # True = analyze outside the mask

In [ ]:
# Just run

def get_channel(image, channel):
    """
    Extract a channel from:
        - Y × X single-channel images
        - C × Y × X channel-first images
        - Y × X × C channel-last images
    """
    image = np.squeeze(image)

    if image.ndim == 2:
        if channel != 0:
            raise ValueError(
                f"Requested channel {channel}, but this is a single-channel "
                f"image with shape {image.shape}."
            )
        return image

    if image.ndim == 3:
        # Channel-first image: C × Y × X
        if image.shape[0] <= 4:
            if channel >= image.shape[0]:
                raise ValueError(
                    f"Requested channel {channel}, but the image only has "
                    f"{image.shape[0]} channel(s). Shape: {image.shape}"
                )
            return image[channel]

        # Channel-last image: Y × X × C
        if image.shape[-1] <= 4:
            if channel >= image.shape[-1]:
                raise ValueError(
                    f"Requested channel {channel}, but the image only has "
                    f"{image.shape[-1]} channel(s). Shape: {image.shape}"
                )
            return image[..., channel]

        raise ValueError(
            f"Ambiguous 3D image shape {image.shape}. The image may be a "
            "Z-stack rather than a multichannel image."
        )

    raise ValueError(
        f"Unsupported image shape {image.shape}. Expected Y×X, C×Y×X, "
        "or Y×X×C."
    )


def blur_and_binary(img, blur=None):
    """
    Apply Gaussian blur to a binary image and convert it back to binary.
    """
    blurred_image = filters.gaussian(
        img.astype(float),
        sigma=blur
    )

    binary_blurred_image = blurred_image > 0.1

    return binary_blurred_image


def show_mask(
    img,
    blur_mask=None,
    second_mask=None,
    blur_mask2=None,
    display_percentile=99.5
):
    image = tifffile.imread(img)

    print(
        f"Loaded: {os.path.basename(img)} | "
        f"shape: {image.shape} | dtype: {image.dtype}"
    )

    # ---------------- Target ----------------
    img_target = get_channel(image, target)
    binary_target = img_target > threshold_value_target

    # ---------------- Mask 1 ----------------
    img_mask = get_channel(image, mask)
    binary_mask = img_mask > threshold_value_mask

    if blur_mask is not None:
        binary_mask = blur_and_binary(
            binary_mask,
            blur=blur_mask
        )

    if invert_mask:
        binary_mask = ~binary_mask

    # ---------------- Optional Mask 2 ----------------
    if second_mask is not None:
        img_second_mask = get_channel(
            image,
            second_mask
        )

        binary_second_mask = (
            img_second_mask > threshold_value_mask2
        )

        if blur_mask2 is not None:
            binary_second_mask = blur_and_binary(
                binary_second_mask,
                blur=blur_mask2
            )

        if invert_second_mask:
            binary_second_mask = ~binary_second_mask

    # Calculate a dynamic vmax for displaying raw channels
    def calculate_vmax(channel):
        vmax = np.percentile(
            channel,
            display_percentile
        )

        if not np.isfinite(vmax) or vmax <= 0:
            vmax = np.max(channel)

        if not np.isfinite(vmax) or vmax <= 0:
            vmax = 1

        return vmax

    # Information for each row
    rows = [
        {
            "row_label": "Target",
            "name": target_name,
            "raw_image": img_target,
            "binary_image": binary_target,
            "threshold": threshold_value_target,
            "inverted": False
        },
        {
            "row_label": "Mask 1",
            "name": mask_name,
            "raw_image": img_mask,
            "binary_image": binary_mask,
            "threshold": threshold_value_mask,
            "inverted": invert_mask
        }
    ]

    if second_mask is not None:
        rows.append(
            {
                "row_label": "Mask 2",
                "name": second_mask_name,
                "raw_image": img_second_mask,
                "binary_image": binary_second_mask,
                "threshold": threshold_value_mask2,
                "inverted": invert_second_mask
            }
        )

    # Create a 2 × 2 or 3 × 2 display
    fig, axes = plt.subplots(
        nrows=len(rows),
        ncols=2,
        figsize=(12, 5 * len(rows)),
        squeeze=False
    )

    for row_number, row in enumerate(rows):
        # Original image
        axes[row_number, 0].imshow(
            row["raw_image"],
            cmap="gray",
            vmin=0,
            vmax=calculate_vmax(row["raw_image"])
        )

        axes[row_number, 0].set_title(
            f'{row["name"]}: original',
            fontsize=18
        )

        # Thresholded image
        axes[row_number, 1].imshow(
            row["binary_image"],
            cmap="gray"
        )

        threshold_title = (
            f'{row["name"]} > {row["threshold"]:.2f}'
        )

        if row["inverted"]:
            threshold_title += " — inverted"

        axes[row_number, 1].set_title(
            threshold_title,
            fontsize=18
        )

        # Row label on the left side
        axes[row_number, 0].set_ylabel(
            row["row_label"],
            fontsize=20,
            fontweight="bold",
            labelpad=25
        )

    # Remove ticks and borders while preserving row labels
    for ax in axes.flat:
        ax.set_xticks([])
        ax.set_yticks([])

        for spine in ax.spines.values():
            spine.set_visible(False)

    fig.suptitle(
        os.path.basename(img),
        fontsize=16,
        y=1.01
    )

    plt.tight_layout()
    plt.show()


def assess_mask(
    img,
    blur_mask=None,
    second_mask=None,
    blur_mask2=None,
    channel=0
):
    image = tifffile.imread(img)
    raw_img = get_channel(image, channel)

    binary_mask = raw_img > threshold_value_mask

    if blur_mask is not None:
        binary_mask = blur_and_binary(
            binary_mask,
            blur=blur_mask
        )

    if invert_mask:
        binary_mask = ~binary_mask

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(16, 8)
    )

    axes[0].imshow(
        raw_img,
        cmap="gray"
    )
    axes[0].set_title(
        "Original Image",
        fontsize=16
    )
    axes[0].axis("off")

    axes[1].imshow(
        binary_mask,
        cmap="gray"
    )

    if invert_mask:
        axes[1].set_title(
            f"Outside {mask_name}",
            fontsize=16
        )
    else:
        axes[1].set_title(
            f"Inside {mask_name}",
            fontsize=16
        )

    axes[1].axis("off")

    plt.tight_layout()
    plt.show()


def auto_threshold(img, second_mask=None):
    image = tifffile.imread(img)

    img_mask = get_channel(image, mask)
    threshold_value_mask_local = filters.threshold_otsu(
        img_mask
    )

    img_target = get_channel(image, target)
    threshold_value_target_local = filters.threshold_otsu(
        img_target
    )

    data = {
        "Image": [img],
        "Mask_Threshold": [threshold_value_mask_local],
        "Target_Threshold": [threshold_value_target_local]
    }

    if second_mask is not None:
        img_second_mask = get_channel(
            image,
            second_mask
        )

        threshold_value_mask2_local = filters.threshold_otsu(
            img_second_mask
        )

        data["Second_Mask_Threshold"] = [
            threshold_value_mask2_local
        ]

    return pd.DataFrame(data)


def process_images(path, second_mask=None):
    all_thresholds = []

    for filename in os.listdir(path):
        if filename.lower().endswith((".tif", ".tiff")):
            image_path = os.path.join(
                path,
                filename
            )

            thresholds = auto_threshold(
                image_path,
                second_mask
            )

            all_thresholds.append(thresholds)

    if not all_thresholds:
        raise ValueError(
            f"No TIFF images were found in:\n{path}"
        )

    combined_df = pd.concat(
        all_thresholds,
        ignore_index=True
    )

    return combined_df


def represent_dataset(df, second_mask=None):
    # Calculate median threshold values
    median_mask_threshold = df[
        "Mask_Threshold"
    ].median()

    median_target_threshold = df[
        "Target_Threshold"
    ].median()

    if second_mask is not None:
        median_second_mask_threshold = df[
            "Second_Mask_Threshold"
        ].median()

    # Find the image whose threshold is closest to the median
    def closest_to_median(df, column, median_value):
        differences = abs(
            df[column] - median_value
        )

        closest_row = df.loc[
            differences.idxmin()
        ]

        return closest_row["Image"]

    closest_mask_threshold_image = closest_to_median(
        df,
        "Mask_Threshold",
        median_mask_threshold
    )

    closest_target_threshold_image = closest_to_median(
        df,
        "Target_Threshold",
        median_target_threshold
    )

    if second_mask is not None:
        closest_second_mask_threshold_image = closest_to_median(
            df,
            "Second_Mask_Threshold",
            median_second_mask_threshold
        )

    print(
        f"Median for Target_Threshold ({target_name}): "
        f"{median_target_threshold}, Closest Image: "
        f'r"{closest_target_threshold_image}"'
    )

    print(
        f"Median for Mask_Threshold ({mask_name}): "
        f"{median_mask_threshold}, Closest Image: "
        f'r"{closest_mask_threshold_image}"'
    )

    if second_mask is not None:
        print(
            f"Median for Second_Mask_Threshold "
            f"({second_mask_name}): "
            f"{median_second_mask_threshold}, Closest Image: "
            f'r"{closest_second_mask_threshold_image}"'
        )

    return closest_target_threshold_image


def stats_test_2_groups(feature, group_col="Group"):
    # Separate the data by group
    groups = df_fin[group_col].unique()

    if len(groups) != 2:
        raise ValueError(
            f"Exactly two groups are required, but found: "
            f"{list(groups)}"
        )

    group_data = {
        group: df_fin.loc[
            df_fin[group_col] == group,
            feature
        ].dropna()
        for group in groups
    }

    # Test each group for normality
    shapiro_results = {
        group: shapiro(values)
        for group, values in group_data.items()
    }

    alpha = 0.05

    if all(
        result.pvalue > alpha
        for result in shapiro_results.values()
    ):
        test_result = ttest_ind(
            *group_data.values()
        )
        test_name = "Unpaired t-test"

    else:
        test_result = mannwhitneyu(
            *group_data.values()
        )
        test_name = "Mann–Whitney U test"

    if test_result.pvalue < 0.001:
        significance = "***"
    elif test_result.pvalue < 0.01:
        significance = "**"
    elif test_result.pvalue < 0.05:
        significance = "*"
    else:
        significance = "ns"

    print(f"Test: {test_name}")

    return round(test_result.pvalue, 3), significance


thresholds_df = process_images(
    path,
    second_mask
)

median_img = represent_dataset(
    thresholds_df,
    second_mask
)

## Choose thresholds for channels. The median image will be displayed below.

In [ ]:
# ============================================================
# Choose thresholds
# ============================================================

# Target channel
threshold_value_target = 50

# First mask
threshold_value_mask = 8
blur_mask = 1

# Optional second mask
threshold_value_mask2 = 30
blur_mask2 = 0


# ============================================================
# Image display settings
# ============================================================

# Sets the display vmax using this intensity percentile.
# This affects only the display--not the measurements.
display_percentile = 99.5

# By default, show the representative image selected above.
img_to_show = median_img

# Alternatively, enter the path to a specific image:
# img_to_show = r"C:\Users\owenf\Desktop\example_image.tif"

show_mask(
    img_to_show,
    blur_mask,
    second_mask,
    blur_mask2,
    display_percentile
)


In [ ]:
# Just run
def process_imgs(
    image_path,
    threshold_value_target,
    threshold_value_mask,
    threshold_value_mask2=None,
    blur_mask=None,
    blur_mask2=None
):
    # Load the image
    image = tifffile.imread(image_path)

    img_target = image[target]
    img_mask = image[mask]

    # ========================================================
    # Create the target threshold
    # ========================================================

    binary_target = img_target > threshold_value_target

    # ========================================================
    # Create Mask 1
    # ========================================================

    binary_mask = img_mask > threshold_value_mask

    if blur_mask is not None:
        binary_mask = blur_and_binary(
            binary_mask,
            blur=blur_mask
        )

    # Warn if very little of the image is positive before inversion
    mask_positive_percent = (
        np.sum(binary_mask) / binary_mask.size
    ) * 100

    if mask_positive_percent < 1:
        print(
            f'The {mask_name} channel has <1% positive pixels in '
            f'r"{image_path}"'
        )

    # Invert Mask 1 if requested
    if invert_mask:
        binary_mask = ~binary_mask

    # Mask 1 initially defines the analysis region
    analysis_mask = binary_mask.copy()

    # ========================================================
    # Create optional Mask 2
    # ========================================================

    if second_mask is not None:
        img_second_mask = image[second_mask]

        binary_second_mask = (
            img_second_mask > threshold_value_mask2
        )

        if blur_mask2 is not None:
            binary_second_mask = blur_and_binary(
                binary_second_mask,
                blur=blur_mask2
            )

        # Warn if very little of the image is positive before inversion
        second_mask_positive_percent = (
            np.sum(binary_second_mask) /
            binary_second_mask.size
        ) * 100

        if second_mask_positive_percent < 1:
            print(
                f'The {second_mask_name} channel has <1% positive '
                f'pixels in r"{image_path}"'
            )

        # Invert Mask 2 if requested
        if invert_second_mask:
            binary_second_mask = ~binary_second_mask

        # The final analysis region is where both masks overlap
        analysis_mask = (
            analysis_mask &
            binary_second_mask
        )

    # ========================================================
    # Whole-image measurements
    # ========================================================

    num_target_positive_pixels = np.sum(binary_target)
    total_image_pixels = binary_target.size

    target_positive_area_percent = (
        num_target_positive_pixels /
        total_image_pixels
    ) * 100

    target_positive_values = img_target[binary_target]

    if target_positive_values.size > 0:
        mean_target_intensity = np.mean(
            target_positive_values
        )
    else:
        mean_target_intensity = 0

    # ========================================================
    # Measurements inside the analysis region
    # ========================================================

    # Target-positive pixels that are also inside the mask(s)
    target_positive_in_masks = (
        binary_target &
        analysis_mask
    )

    masked_target_values = img_target[
        target_positive_in_masks
    ]

    if masked_target_values.size > 0:
        mean_target_intensity_in_masks = np.mean(
            masked_target_values
        )
    else:
        mean_target_intensity_in_masks = 0

    # Percentage of the masked region occupied by
    # target-positive pixels
    num_pixels_in_masks = np.sum(analysis_mask)
    num_target_positive_pixels_in_masks = np.sum(
        target_positive_in_masks
    )

    if num_pixels_in_masks > 0:
        target_positive_area_in_masks = (
            num_target_positive_pixels_in_masks /
            num_pixels_in_masks
        ) * 100
    else:
        target_positive_area_in_masks = 0

        print(
            f'Warning: the final analysis mask contains no pixels in '
            f'r"{image_path}"'
        )

    return (
        mean_target_intensity,
        target_positive_area_percent,
        mean_target_intensity_in_masks,
        target_positive_area_in_masks
    )


# ============================================================
# Process every image
# ============================================================

results = []

for filename in sorted(os.listdir(path)):
    if filename.lower().endswith((".tif", ".tiff")):
        image_path = os.path.join(
            path,
            filename
        )

        (
            mean_target_intensity,
            target_positive_area_percent,
            mean_target_intensity_in_masks,
            target_positive_area_in_masks
        ) = process_imgs(
            image_path,
            threshold_value_target,
            threshold_value_mask,
            threshold_value_mask2,
            blur_mask,
            blur_mask2
        )

        image_results = {
            "Filename": filename,

            f"Target Threshold ({target_name})":
                threshold_value_target,

            "Mask 1 Threshold ({mask_name})":
                threshold_value_mask,

            "Mask 1 Inverted":
                invert_mask,

            "Mean Target Intensity Above Threshold (Whole Image)":
                mean_target_intensity,

            "Target-Positive Area (% of Whole Image)":
                target_positive_area_percent,

            "Mean Target Intensity Above Threshold in Mask(s)":
                mean_target_intensity_in_masks,

            "Target-Positive Area (% of Masked Region)":
                target_positive_area_in_masks
        }

        # Only add Mask 2 information when it is enabled
        if second_mask is not None:
            image_results.update({
                "Mask 2 Threshold ({second_mask_name})":
                    threshold_value_mask2,

                "Mask 2 Inverted":
                    invert_second_mask
            })

        results.append(image_results)


# Create the final results table
df_fin = pd.DataFrame(results)

df_fin

In [ ]:
# ============================================================
# Save results as a CSV
# ============================================================

# Optional label added to the beginning of the filename
file_suffix = ""

# Describe whether the analysis is inside or outside Mask 1
mask_description = (
    f"Outside {mask_name}"
    if invert_mask
    else f"Inside {mask_name}"
)

# Build the output filename
title_parts = [
    file_suffix,
    f"{target_name} Thresh-{threshold_value_target}",
    f"{mask_description} Thresh-{threshold_value_mask}"
]

# Add Mask 2 information only when it is enabled
if second_mask is not None:
    second_mask_description = (
        f"Outside {second_mask_name}"
        if invert_second_mask
        else f"Inside {second_mask_name}"
    )

    title_parts.append(
        f"{second_mask_description} "
        f"Thresh-{threshold_value_mask2}"
    )

# Remove any empty parts and join the filename together
csv_title = " - ".join(
    part for part in title_parts if part
)

# Create an output folder inside the image folder
output_folder = os.path.join(
    path,
    "Analysis Results"
)

os.makedirs(
    output_folder,
    exist_ok=True
)

# Create the complete output path
csv_path = os.path.join(
    output_folder,
    csv_title + ".csv"
)

# Save the results
df_fin.to_csv(
    csv_path,
    index=False
)

print(f"Results saved to:\n{csv_path}")